# GST Normalisation and Model Comparison

**Runtime:** T4 GPU. Select **Runtime > Run all**.

This standalone notebook:

1. reads every supported image in `gold_standard_testing/`;
2. creates three versions of each page:
   - original greyscale;
   - background-normalised;
   - background-normalised plus CLAHE;
3. runs the same trained YOLO segmentation model against all three versions using overlapping tiles;
4. recombines tile predictions onto the complete page;
5. saves individual prediction images and three-way comparison images;
6. writes a JSON and CSV summary.

This experiment does **not retrain the model**. It isolates whether page normalisation improves inference.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics

## Configuration

By default the notebook automatically selects the most recently modified `best.pt` under `training_runs/`. To test a specific model, put its full path in `MODEL_PATH_OVERRIDE`.

In [ ]:
import cv2, json, csv
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import torch
from ultralytics import YOLO

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CONFIG_PATH=ROOT/'project_config.json'
GST_DIR=ROOT/'gold_standard_testing'
OUTPUT_ROOT=ROOT/'gold_standard_normalisation_tests'
TRAINING_RUNS_DIR=ROOT/'training_runs'

# Leave as None to choose the newest best.pt automatically.
MODEL_PATH_OVERRIDE=None

# Normalisation parameters
BACKGROUND_BLUR_SIGMA=75
CLAHE_CLIP_LIMIT=2.0
CLAHE_GRID_SIZE=(8,8)

# Prediction parameters
CONFIDENCE=0.10
TILE_SIZE=640
TILE_STRIDE=320
MODEL_IMAGE_SIZE=640

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff'}
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR=OUTPUT_ROOT/RUN_NAME
NORMALISED_DIR=RUN_DIR/'normalised_pages'
PREDICTION_DIR=RUN_DIR/'predictions'
COMPARISON_DIR=RUN_DIR/'comparisons'
for folder in [GST_DIR,RUN_DIR,NORMALISED_DIR,PREDICTION_DIR,COMPARISON_DIR]:
    folder.mkdir(parents=True,exist_ok=True)

if MODEL_PATH_OVERRIDE:
    MODEL_PATH=Path(MODEL_PATH_OVERRIDE)
else:
    candidates=sorted(TRAINING_RUNS_DIR.rglob('best.pt'),key=lambda p:p.stat().st_mtime,reverse=True)
    if not candidates:
        raise RuntimeError('No best.pt model found under training_runs/.')
    MODEL_PATH=candidates[0]

if not MODEL_PATH.exists():
    raise RuntimeError(f'Model not found: {MODEL_PATH}')
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then Run all again.')

GST_FILES=sorted([p for p in GST_DIR.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED])
if not GST_FILES:
    raise RuntimeError(f'No GST images found in {GST_DIR}')

print('GPU:',torch.cuda.get_device_name(0))
print('Model:',MODEL_PATH)
print('GST files:',len(GST_FILES))
for path in GST_FILES: print(' -',path.name)
print('Output:',RUN_DIR)

## Create original and normalised page versions

Background division reduces page tint, shadows and scanner gradients. CLAHE then increases local contrast. No binary threshold is applied because thresholding could remove faint pencil.

In [ ]:
def normalise_background(image,blur_sigma=BACKGROUND_BLUR_SIGMA):
    background=cv2.GaussianBlur(image,(0,0),blur_sigma)
    background=np.maximum(background,1)
    return cv2.divide(image,background,scale=255)

def apply_clahe(image):
    clahe=cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT,tileGridSize=CLAHE_GRID_SIZE)
    return clahe.apply(image)

page_versions={}
for source_path in GST_FILES:
    original=cv2.imread(str(source_path),cv2.IMREAD_GRAYSCALE)
    if original is None:
        print('Unreadable:',source_path); continue
    background_normalised=normalise_background(original)
    clahe_normalised=apply_clahe(background_normalised)
    versions={'original':original,'bgnorm':background_normalised,'clahe':clahe_normalised}
    page_versions[source_path.name]=versions
    page_folder=NORMALISED_DIR/source_path.stem
    page_folder.mkdir(parents=True,exist_ok=True)
    for version_name,image in versions.items():
        cv2.imwrite(str(page_folder/f'{source_path.stem}_{version_name}.png'),image)

print('Prepared',len(page_versions),'pages and',sum(len(v) for v in page_versions.values()),'page versions.')

## Preview normalisation

The display shows the first few pages before predictions are run.

In [ ]:
for file_name,versions in list(page_versions.items())[:4]:
    plt.figure(figsize=(18,7))
    for index,(name,image) in enumerate(versions.items(),1):
        plt.subplot(1,3,index); plt.imshow(image,cmap='gray'); plt.title(f'{file_name}: {name}'); plt.axis('off')
    plt.tight_layout(); plt.show()

## Tiled YOLO inference

Each full-resolution page is divided into overlapping 640 × 640 tiles. The strongest prediction confidence is retained for overlapping pixels, then predictions are recombined onto the complete page.

In [ ]:
model=YOLO(str(MODEL_PATH))

def tile_positions(length):
    if length<=TILE_SIZE: return [0]
    positions=list(range(0,length-TILE_SIZE+1,TILE_STRIDE))
    if positions[-1]!=length-TILE_SIZE: positions.append(length-TILE_SIZE)
    return positions

def tiled_predict(gray_image):
    colour_image=cv2.cvtColor(gray_image,cv2.COLOR_GRAY2BGR)
    height,width=gray_image.shape
    padded_height=max(height,TILE_SIZE); padded_width=max(width,TILE_SIZE)
    canvas=np.full((padded_height,padded_width,3),255,np.uint8)
    canvas[:height,:width]=colour_image
    confidence_map=np.zeros((padded_height,padded_width),np.float32)
    detection_count=0

    for y in tile_positions(padded_height):
        for x in tile_positions(padded_width):
            tile=canvas[y:y+TILE_SIZE,x:x+TILE_SIZE]
            result=model.predict(tile,conf=CONFIDENCE,imgsz=MODEL_IMAGE_SIZE,device=0,verbose=False,retina_masks=True)[0]
            if result.masks is None: continue
            masks=result.masks.data.cpu().numpy()
            confidences=result.boxes.conf.cpu().numpy() if result.boxes is not None else np.ones(len(masks))
            detection_count+=len(masks)
            for mask,confidence in zip(masks,confidences):
                mask=cv2.resize(mask,(TILE_SIZE,TILE_SIZE),interpolation=cv2.INTER_NEAREST)>0.5
                region=confidence_map[y:y+TILE_SIZE,x:x+TILE_SIZE]
                update=mask & (confidence>region)
                region[update]=confidence

    confidence_map=confidence_map[:height,:width]
    selected=confidence_map>=CONFIDENCE
    highlighted=colour_image.copy()
    highlighted[selected]=(0,0,255)
    prediction=cv2.addWeighted(colour_image,.68,highlighted,.32,0)
    return prediction,confidence_map,detection_count

summary=[]
for file_name,versions in page_versions.items():
    stem=Path(file_name).stem
    page_prediction_dir=PREDICTION_DIR/stem
    page_prediction_dir.mkdir(parents=True,exist_ok=True)
    version_outputs={}
    for version_name,image in versions.items():
        prediction,confidence_map,detection_count=tiled_predict(image)
        predicted_pixels=int(np.sum(confidence_map>=CONFIDENCE))
        mean_confidence=float(confidence_map[confidence_map>=CONFIDENCE].mean()) if predicted_pixels else 0.0
        max_confidence=float(confidence_map.max())
        output_path=page_prediction_dir/f'{stem}_{version_name}_prediction.png'
        cv2.imwrite(str(output_path),prediction)
        version_outputs[version_name]=prediction
        summary.append({'file':file_name,'version':version_name,'tile_detections':int(detection_count),'predicted_pixels':predicted_pixels,'mean_predicted_confidence':mean_confidence,'maximum_confidence':max_confidence,'prediction_file':str(output_path)})
        print(file_name,version_name,'detections',detection_count,'pixels',predicted_pixels,'max confidence',round(max_confidence,4))

    # Three-way comparison at original page resolution
    comparison=np.hstack([version_outputs['original'],version_outputs['bgnorm'],version_outputs['clahe']])
    cv2.imwrite(str(COMPARISON_DIR/f'{stem}_comparison.png'),comparison)

print('Inference complete.')

## Display comparison outputs

Each image displays original-input predictions, background-normalised predictions and CLAHE predictions from left to right.

In [ ]:
comparison_files=sorted(COMPARISON_DIR.glob('*_comparison.png'))
for path in comparison_files:
    image=cv2.imread(str(path))
    plt.figure(figsize=(22,12)); plt.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB)); plt.title(path.name+' | original | background normalised | CLAHE'); plt.axis('off'); plt.show()

## Save experiment summary

Lower predicted-pixel counts are not automatically better. A useful normalisation should reduce texture and printed-notation false positives while retaining genuine handwriting.

In [ ]:
json_path=RUN_DIR/'normalisation_summary.json'
csv_path=RUN_DIR/'normalisation_summary.csv'
json_path.write_text(json.dumps({'model':str(MODEL_PATH),'confidence':CONFIDENCE,'tile_size':TILE_SIZE,'tile_stride':TILE_STRIDE,'background_blur_sigma':BACKGROUND_BLUR_SIGMA,'clahe_clip_limit':CLAHE_CLIP_LIMIT,'results':summary},indent=2))
with open(csv_path,'w',newline='',encoding='utf-8') as csv_file:
    fields=list(summary[0].keys())
    writer=csv.DictWriter(csv_file,fieldnames=fields); writer.writeheader(); writer.writerows(summary)
print('Predictions:',PREDICTION_DIR)
print('Comparisons:',COMPARISON_DIR)
print('JSON summary:',json_path)
print('CSV summary:',csv_path)